In [0]:
%sql
USE CATALOG credlake;
USE SCHEMA silver;

SELECT
    current_catalog() AS current_catalog,
    current_schema() AS current_schema;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT contract_id) AS distinct_contract_ids,
    MIN(principal_amount) AS min_principal,
    MAX(principal_amount) AS max_principal,
    MIN(annual_interest_rate) AS min_interest_rate,
    MAX(annual_interest_rate) AS max_interest_rate,
    MIN(term_months) AS min_term,
    MAX(term_months) AS max_term,
    MIN(contract_date) AS min_contract_date,
    MAX(contract_date) AS max_contract_date
FROM credlake.bronze.contracts_raw;

In [0]:
%sql
SELECT
    contract_status,
    COUNT(*) AS total
FROM credlake.bronze.contracts_raw
GROUP BY contract_status
ORDER BY total DESC;

In [0]:
%sql
SELECT
    product_id,
    product_name,
    allowed_customer_type,
    max_term_months
FROM credlake.bronze.products_raw
ORDER BY product_id;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS credlake.silver.contracts (
    contract_id                        BIGINT,
    customer_id                        BIGINT,
    product_id                         INT,
    contract_date                      DATE,
    principal_amount                   DECIMAL(18,2),
    annual_interest_rate               DECIMAL(10,4),
    term_months                        INT,
    contract_status                    STRING,
    source_updated_at                  TIMESTAMP,
    source_system                      STRING,
    source_batch_id                    STRING,
    customer_snapshot_date             DATE,
    source_file_path                   STRING,
    source_file_name                   STRING,
    source_file_modification_time      TIMESTAMP,
    silver_processed_at                TIMESTAMP,
    record_hash                        STRING
)
USING DELTA
COMMENT 'Contratos financeiros deduplicados e aprovados pelas regras de qualidade.'
TBLPROPERTIES (
    'data_layer' = 'silver',
    'data_domain' = 'credit',
    'contains_synthetic_data' = 'true'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS credlake.silver.contracts_quarantine (
    quarantine_id                      STRING,
    contract_id                        BIGINT,
    customer_id                        BIGINT,
    product_id                         INT,
    contract_date                      DATE,
    principal_amount                   DECIMAL(18,2),
    annual_interest_rate               DECIMAL(10,4),
    term_months                        INT,
    contract_status                    STRING,
    source_updated_at                  TIMESTAMP,
    source_system                      STRING,
    source_batch_id                    STRING,
    source_file_path                   STRING,
    source_file_name                   STRING,
    source_file_modification_time      TIMESTAMP,
    duplicate_rank                     INT,
    error_codes                        ARRAY<STRING>,
    error_reasons                      ARRAY<STRING>,
    quarantined_at                     TIMESTAMP,
    record_hash                        STRING
)
USING DELTA
COMMENT 'Contratos rejeitados pela Silver, preservados com os respectivos motivos.'
TBLPROPERTIES (
    'data_layer' = 'silver_quarantine',
    'data_domain' = 'credit',
    'contains_synthetic_data' = 'true'
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_contracts_classified AS

WITH ranked_contracts AS (
    SELECT
        b.*,
        ROW_NUMBER() OVER (
            PARTITION BY b.contract_id
            ORDER BY
                b.source_updated_at DESC NULLS LAST,
                b.source_file_modification_time DESC NULLS LAST,
                b.source_file_path DESC,
                b.source_batch_id DESC
        ) AS duplicate_rank

    FROM credlake.bronze.contracts_raw AS b
),

latest_customers AS (
    SELECT
        customer_id,
        customer_type,
        snapshot_date
    FROM (
        SELECT
            customer_id,
            customer_type,
            snapshot_date,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY
                    snapshot_date DESC,
                    updated_at DESC NULLS LAST,
                    ingested_at DESC
            ) AS customer_rank

        FROM credlake.bronze.customers_raw
        WHERE corrupt_record IS NULL
    )
    WHERE customer_rank = 1
),

latest_products AS (
    SELECT
        product_id,
        product_name,
        allowed_customer_type,
        max_term_months
    FROM (
        SELECT
            product_id,
            product_name,
            allowed_customer_type,
            max_term_months,
            ROW_NUMBER() OVER (
                PARTITION BY product_id
                ORDER BY
                    source_file_modification_time DESC NULLS LAST,
                    ingested_at DESC
            ) AS product_rank

        FROM credlake.bronze.products_raw
    )
    WHERE product_rank = 1
),

enriched_contracts AS (
    SELECT
        c.*,
        customer.customer_id AS matched_customer_id,
        customer.snapshot_date AS customer_snapshot_date,
        product.product_id AS matched_product_id,
        product.max_term_months AS product_max_term_months

    FROM ranked_contracts AS c

    LEFT JOIN latest_customers AS customer
        ON c.customer_id = customer.customer_id

    LEFT JOIN latest_products AS product
        ON c.product_id = product.product_id
)

SELECT
    *,

    FILTER(
        ARRAY(
            CASE
                WHEN duplicate_rank > 1
                THEN 'DUPLICATE_CONTRACT_ID'
            END,
            CASE
                WHEN contract_id IS NULL OR contract_id <= 0
                THEN 'INVALID_CONTRACT_ID'
            END,
            CASE
                WHEN customer_id IS NULL
                THEN 'NULL_CUSTOMER_ID'
            END,
            CASE
                WHEN customer_id IS NOT NULL
                     AND matched_customer_id IS NULL
                THEN 'CUSTOMER_NOT_FOUND'
            END,
            CASE
                WHEN product_id IS NULL
                THEN 'NULL_PRODUCT_ID'
            END,
            CASE
                WHEN product_id IS NOT NULL
                     AND matched_product_id IS NULL
                THEN 'PRODUCT_NOT_FOUND'
            END,
            CASE
                WHEN principal_amount IS NULL
                     OR principal_amount <= 0
                THEN 'INVALID_PRINCIPAL_AMOUNT'
            END,
            CASE
                WHEN annual_interest_rate IS NULL
                     OR annual_interest_rate <= 0
                     OR annual_interest_rate > 1
                THEN 'INVALID_INTEREST_RATE'
            END,
            CASE
                WHEN term_months IS NULL
                     OR term_months <= 0
                THEN 'INVALID_TERM'
            END,
            CASE
                WHEN product_max_term_months IS NOT NULL
                     AND term_months > product_max_term_months
                THEN 'TERM_EXCEEDS_PRODUCT_LIMIT'
            END,
            CASE
                WHEN contract_status IS NULL
                     OR TRIM(contract_status) = ''
                THEN 'INVALID_CONTRACT_STATUS'
            END,
            CASE
                WHEN contract_date IS NULL
                     OR contract_date > CURRENT_DATE()
                THEN 'INVALID_CONTRACT_DATE'
            END
        ),
        error_code -> error_code IS NOT NULL
    ) AS error_codes,

    FILTER(
        ARRAY(
            CASE
                WHEN duplicate_rank > 1
                THEN 'Existe outro registro priorizado para o mesmo contract_id'
            END,
            CASE
                WHEN contract_id IS NULL OR contract_id <= 0
                THEN 'O identificador do contrato deve ser positivo e não nulo'
            END,
            CASE
                WHEN customer_id IS NULL
                THEN 'O identificador do cliente é obrigatório'
            END,
            CASE
                WHEN customer_id IS NOT NULL
                     AND matched_customer_id IS NULL
                THEN 'O cliente não existe no snapshot mais recente'
            END,
            CASE
                WHEN product_id IS NULL
                THEN 'O identificador do produto é obrigatório'
            END,
            CASE
                WHEN product_id IS NOT NULL
                     AND matched_product_id IS NULL
                THEN 'O produto não foi encontrado no cadastro'
            END,
            CASE
                WHEN principal_amount IS NULL
                     OR principal_amount <= 0
                THEN 'O valor principal deve ser maior que zero'
            END,
            CASE
                WHEN annual_interest_rate IS NULL
                     OR annual_interest_rate <= 0
                     OR annual_interest_rate > 1
                THEN 'A taxa anual deve estar no intervalo maior que zero e menor ou igual a um'
            END,
            CASE
                WHEN term_months IS NULL
                     OR term_months <= 0
                THEN 'O prazo deve possuir pelo menos um mês'
            END,
            CASE
                WHEN product_max_term_months IS NOT NULL
                     AND term_months > product_max_term_months
                THEN 'O prazo do contrato ultrapassa o máximo permitido pelo produto'
            END,
            CASE
                WHEN contract_status IS NULL
                     OR TRIM(contract_status) = ''
                THEN 'O status do contrato é obrigatório'
            END,
            CASE
                WHEN contract_date IS NULL
                     OR contract_date > CURRENT_DATE()
                THEN 'A data do contrato é nula ou está no futuro'
            END
        ),
        reason -> reason IS NOT NULL
    ) AS error_reasons

FROM enriched_contracts;

In [0]:
%sql
SELECT
    COUNT(*) AS total_bronze_rows,

    SUM(
        CASE WHEN SIZE(error_codes) = 0 THEN 1 ELSE 0 END
    ) AS valid_rows,

    SUM(
        CASE WHEN SIZE(error_codes) > 0 THEN 1 ELSE 0 END
    ) AS quarantined_rows

FROM vw_contracts_classified;

In [0]:
%sql
SELECT
    error_code,
    COUNT(*) AS occurrences
FROM vw_contracts_classified
LATERAL VIEW EXPLODE(error_codes) errors AS error_code
GROUP BY error_code
ORDER BY occurrences DESC, error_code;

In [0]:
%sql
SELECT
    contract_id,
    customer_id,
    principal_amount,
    duplicate_rank,
    error_codes,
    error_reasons
FROM vw_contracts_classified
WHERE SIZE(error_codes) > 0
ORDER BY contract_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_valid_contracts AS

SELECT
    contract_id,
    customer_id,
    product_id,
    contract_date,
    principal_amount,
    annual_interest_rate,
    term_months,
    contract_status,
    source_updated_at,
    source_system,
    source_batch_id,
    customer_snapshot_date,
    source_file_path,
    source_file_name,
    source_file_modification_time,
    CURRENT_TIMESTAMP() AS silver_processed_at,
    SHA2(
        TO_JSON(
            NAMED_STRUCT(
                'contract_id', contract_id,
                'customer_id', customer_id,
                'product_id', product_id,
                'contract_date', contract_date,
                'principal_amount', principal_amount,
                'annual_interest_rate', annual_interest_rate,
                'term_months', term_months,
                'contract_status', contract_status,
                'source_updated_at', source_updated_at
            )
        ),
        256
    ) AS record_hash
FROM vw_contracts_classified
WHERE SIZE(error_codes) = 0;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_quarantined_contracts AS

SELECT
    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(CAST(contract_id AS STRING), 'NULL'),
            COALESCE(source_file_path, 'NULL'),
            COALESCE(source_batch_id, 'NULL'),
            CAST(duplicate_rank AS STRING)
        ),
        256
    ) AS quarantine_id,
    contract_id,
    customer_id,
    product_id,
    contract_date,
    principal_amount,
    annual_interest_rate,
    term_months,
    contract_status,
    source_updated_at,
    source_system,
    source_batch_id,
    source_file_path,
    source_file_name,
    source_file_modification_time,
    duplicate_rank,
    error_codes,
    error_reasons,
    CURRENT_TIMESTAMP() AS quarantined_at,
    SHA2(
        TO_JSON(
            NAMED_STRUCT(
                'contract_id', contract_id,
                'customer_id', customer_id,
                'product_id', product_id,
                'principal_amount', principal_amount,
                'duplicate_rank', duplicate_rank,
                'error_codes', error_codes
            )
        ),
        256
    ) AS record_hash

FROM vw_contracts_classified
WHERE SIZE(error_codes) > 0;

In [0]:
%sql
MERGE INTO credlake.silver.contracts AS target

USING vw_valid_contracts AS source

ON target.contract_id = source.contract_id

WHEN MATCHED
     AND target.record_hash <> source.record_hash
THEN UPDATE SET *

WHEN NOT MATCHED
THEN INSERT *

WHEN NOT MATCHED BY SOURCE
THEN DELETE;

In [0]:
%sql

MERGE INTO credlake.silver.contracts_quarantine AS target

USING vw_quarantined_contracts AS source

ON target.quarantine_id = source.quarantine_id

WHEN MATCHED
     AND target.record_hash <> source.record_hash
THEN UPDATE SET *

WHEN NOT MATCHED
THEN INSERT *

WHEN NOT MATCHED BY SOURCE
THEN DELETE;

In [0]:
%sql
SELECT
    'silver.contracts' AS dataset,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT contract_id) AS distinct_business_keys
FROM credlake.silver.contracts

UNION ALL

SELECT
    'silver.contracts_quarantine',
    COUNT(*),
    COUNT(DISTINCT quarantine_id)
FROM credlake.silver.contracts_quarantine;